# Streamlit App — Demo / Reference Notebook

This notebook is a **Colab-runnable reference** version of `app.py`.
It is intended for demo and experimentation purposes only.
For the production app, use `app.py` with `streamlit run app.py`.

> **Note:** The cell below will show `ScriptRunContext` warnings when run outside Streamlit — this is expected and can be ignored.

In [ ]:
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np
import os
import hashlib

# Page config
st.set_page_config(page_title='Fashion MNIST Classifier', layout='centered')

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

@st.cache_resource
def load_model():
    model_path = 'fashion_mnist_cnn.keras'
    if not os.path.exists(model_path):
        return None
    try:
        return tf.keras.models.load_model(model_path)
    except Exception as e:
        st.error(f'Failed to load model: {e}')
        return None

model = load_model()
if model is None:
    st.error("Model file not found. Run the training notebook first.")
    st.stop()

st.title('Fashion MNIST Image Classifier')
uploaded_file = st.file_uploader('Choose an image...', type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    if uploaded_file.size > 5 * 1024 * 1024:
        st.error('File too large. Please upload an image under 5MB.')
        st.stop()
    image = Image.open(uploaded_file).convert('L')
    st.image(image, caption='Uploaded image', width=200)
    img_array = np.array(image.resize((28, 28))).astype('float32') / 255.0
    corners = np.concatenate([img_array[:3,:3].flatten(), img_array[:3,-3:].flatten(),
                               img_array[-3:,:3].flatten(), img_array[-3:,-3:].flatten()])
    if corners.mean() > 0.5:
        img_array = 1.0 - img_array
    input_tensor = np.expand_dims(np.expand_dims(img_array, -1), 0)
    img_hash = hashlib.md5(uploaded_file.getvalue()).hexdigest()
    if st.session_state.get('last_hash') != img_hash:
        with st.spinner('Classifying...'):
            predictions = model.predict(input_tensor, verbose=0)
        st.session_state['last_hash'] = img_hash
        st.session_state['predictions'] = predictions
    else:
        predictions = st.session_state['predictions']
    idx = int(np.argmax(predictions[0]))
    st.subheader('Prediction')
    st.write(f"Class: **{CLASS_NAMES[idx]}**")
    st.write(f"Confidence: **{predictions[0][idx]*100:.1f}%**")
    st.subheader('All Class Probabilities')
    st.bar_chart({CLASS_NAMES[i]: float(predictions[0][i]) for i in range(len(CLASS_NAMES))})

In [ ]:
# Save model after training (run this after cnn_algorithm_fashion_mnist.ipynb)
# model.save('fashion_mnist_cnn.keras')